### MIG Cement Demand Forecasting

#### Data Understanding

This notebook represents the **Data Understanding** stage of the CRISP-DM workflow.

The objective is to understand the operational data available for forecasting cement demand across Midlands Infrastructure Group (MIG) construction sites before cleaning, feature engineering, or modelling begins.

The analysis focuses on four questions:

* What data is available?
* What does each dataset represent?
* What should the forecasting model predict?
* Are there structural or quality issues that must be addressed before analysis?



####  Data Snapshot

The MIG SQLite database contains three related tables:

* `Operations`
* `Sites`
* `CementTypes`

The main `Operations` dataset contains:

**32,880 daily operational records × 11 variables**

The data combines information on:

* planned cement demand,
* actual cement consumption,
* inventory and deliveries,
* weather conditions, and
* silo capacity.

Together, these variables describe how cement moves through each construction site over time.



#### Forecasting Target

The primary forecasting target is:

#### `consumed_tonnes`

This represents the **actual quantity of cement used at a site**.

The central forecasting question is therefore:

> **How much cement will each MIG site require in the future?**

Planned pours, historical consumption, site characteristics, cement type, weather, and time-based patterns may help explain future demand.



#### Operational Logic

The dataset also contains an important inventory relationship:

**Closing Inventory = Opening Inventory + Deliveries − Consumption**

This provides a useful operational consistency check and later connects the demand forecast to MIG's inventory and reorder decision system.

Importantly, demand forecasting and inventory management remain separate tasks:

**Historical Operations → Demand Forecast → Inventory Projection → Risk Assessment → Reorder Decision**



#### Initial Data Understanding

The dataset provides sufficient operational history to investigate cement demand across:

* construction sites,
* cement types,
* time periods,
* planned pours,
* weather conditions, and
* inventory positions.

At this stage, `consumed_tonnes` has been identified as the forecasting target, while the remaining variables provide potential explanatory, contextual, or inventory-management information.

Before exploratory analysis or model development, the dataset must first be validated for data-quality and operational consistency.



#### Data Understanding Summary

The Data Understanding stage confirms that MIG's operational database contains the core information required to build a site-level cement demand forecasting system.

The data connects **planned construction activity, actual cement demand, inventory movement, weather, and physical storage constraints**.

The next CRISP-DM step is:

#### Data Cleaning and Validation

This will verify:

* missing and duplicate records,
* invalid values,
* inventory balance consistency,
* silo-capacity violations,
* unusual zero values, and
* consistency across the three database tables.

Once validated, the dataset will be ready for exploratory analysis.


In [1]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect("../data/MIG_Cement_Records.db")

In [3]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,Sites
1,CementTypes
2,Operations


In [4]:
df = pd.read_sql_query(
    "SELECT * FROM Operations",
    conn
)

df.head()

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448


In [5]:
52.56 + 45.83 -63.85

34.54

In [6]:
52.56 + 45.83 -34.54

63.85

In [7]:
52.56 + 45.83 - 34.54

63.85

In [8]:
df['cement_type'].value_counts()

cement_type
CEM_II     11097
CEM_I      10913
CEM_III    10870
Name: count, dtype: int64

In [9]:
df.to_csv("../data/dataset.csv", index=False)

In [10]:
site_df = pd.read_sql_query(
    "SELECT * FROM Sites",
    conn
)

In [11]:
site_df.to_csv("../data/site_data.csv", index=False)

In [12]:
site_df

,site_id,region,silo_capacity,behavior
0,SITE_001,North,448,aggressive
1,SITE_002,South,288,conservative
2,SITE_003,East,314,aggressive
3,SITE_004,South,472,conservative
4,SITE_005,South,230,aggressive
5,SITE_006,East,443,chaotic
6,SITE_007,East,485,aggressive
7,SITE_008,West,260,aggressive
8,SITE_009,East,352,conservative
9,SITE_010,West,158,aggressive


In [13]:
#Merge Site table (region and behavior) with Operations table to get the region and behavior for each operation
merged_df = pd.merge(df, site_df[[ 'site_id', 'region', 'behavior']], on='site_id', how='left')


In [14]:
merged_df.head()

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,region,behavior
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448,North,aggressive
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448,North,aggressive
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448,North,aggressive
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448,North,aggressive
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448,North,aggressive


In [15]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32880 entries, 0 to 32879
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      32880 non-null  str    
 1   site_id                   32880 non-null  str    
 2   cement_type               32880 non-null  str    
 3   planned_pour_tonnes       32880 non-null  float64
 4   consumed_tonnes           32880 non-null  float64
 5   opening_inventory_tonnes  32880 non-null  float64
 6   deliveries_tonnes         32880 non-null  float64
 7   closing_inventory_tonnes  32880 non-null  float64
 8   rain_mm                   32880 non-null  float64
 9   avg_temp_c                32880 non-null  float64
 10  silo_capacity             32880 non-null  int64  
 11  region                    32880 non-null  str    
 12  behavior                  32880 non-null  str    
dtypes: float64(7), int64(1), str(5)
memory usage: 3.3 MB


In [16]:
merged_df.to_csv("../data/complete_dataset.csv", index=False)

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32880 entries, 0 to 32879
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      32880 non-null  str    
 1   site_id                   32880 non-null  str    
 2   cement_type               32880 non-null  str    
 3   planned_pour_tonnes       32880 non-null  float64
 4   consumed_tonnes           32880 non-null  float64
 5   opening_inventory_tonnes  32880 non-null  float64
 6   deliveries_tonnes         32880 non-null  float64
 7   closing_inventory_tonnes  32880 non-null  float64
 8   rain_mm                   32880 non-null  float64
 9   avg_temp_c                32880 non-null  float64
 10  silo_capacity             32880 non-null  int64  
dtypes: float64(7), int64(1), str(3)
memory usage: 2.8 MB


In [18]:
pd.read_sql_query(
    "SELECT * FROM CementTypes",
    conn
)

,cement_type
0,CEM_I
1,CEM_II
2,CEM_III


In [19]:
site_df=pd.read_sql_query(
    "SELECT * FROM Sites",
    conn
)
site_df.head()

,site_id,region,silo_capacity,behavior
0,SITE_001,North,448,aggressive
1,SITE_002,South,288,conservative
2,SITE_003,East,314,aggressive
3,SITE_004,South,472,conservative
4,SITE_005,South,230,aggressive


#### The major questions will are trying to answer, these questions will drive the whole analysis:

- How much cement does each site consume?

- How does consumption vary by cement type?

- Does cement demand have trends or seasonality?

- How closely does actual consumption follow planned pours?

- Does rain significantly affect consumption?

- Does temperature matter?

- Which sites experience the most volatile demand?

- How frequently do sites approach zero inventory?

- How efficiently are silos being utilized?

- Can historical demand predict future demand?

- Are demand patterns different enough across sites that we need separate models?

- How far ahead can we forecast reliably?

In [20]:
df.groupby("cement_type")["consumed_tonnes"].agg(
    observations="count",
    total_consumption="sum",
    average_daily_consumption="mean",
    median_daily_consumption="median"
)

,observations,total_consumption,average_daily_consumption,median_daily_consumption
cement_type,,,,
CEM_I,10913,259954.71,23.820646,19.79
CEM_II,11097,263418.26,23.737790,19.65
CEM_III,10870,256556.24,23.602230,19.74


In [21]:
df.groupby("site_id")["consumed_tonnes"].agg(
    observations="count",
    total_consumption="sum",
    average_consumption="mean"
).sort_values(
    "total_consumption",
    ascending=False
)

,observations,total_consumption,average_consumption
site_id,,,
SITE_025,1096,33604.06,30.660639
SITE_010,1096,33579.76,30.638467
SITE_018,1096,33348.09,30.427089
SITE_001,1096,33056.40,30.160949
SITE_021,1096,33009.68,30.118321
SITE_005,1096,32935.68,30.050803
SITE_022,1096,32934.03,30.049297
SITE_008,1096,32689.50,29.826186
SITE_007,1096,32607.65,29.751505


In [22]:
# 1. Convert date
df["date"] = pd.to_datetime(df["date"])

# 2. Date range
print("Start:", df["date"].min())
print("End:", df["date"].max())

# 3. Number of sites
print("Number of sites:", df["site_id"].nunique())

# 4. Cement type counts
print(df["cement_type"].value_counts())

# 5. Duplicate site-date-cement records
print(
    "Duplicates:",
    df.duplicated(
        subset=["date", "site_id", "cement_type"]
    ).sum()
)

Start: 2022-01-01 00:00:00
End: 2024-12-31 00:00:00
Number of sites: 30
cement_type
CEM_II     11097
CEM_I      10913
CEM_III    10870
Name: count, dtype: int64
Duplicates: 0


In [23]:
invalid_sites = set(df["site_id"]) - set(site_df["site_id"])

invalid_sites

set()